# K-means trên RFM: chia khách thành nhóm và đặt tên cho đội CRM

**Bối cảnh.** Head of CRM của cửa hàng quà tặng online muốn thiết kế 3–5 chương trình chăm sóc khác nhau cho năm 2012
và hỏi: *"Khách của chúng ta chia thành những nhóm nào? Mỗi nhóm bao nhiêu khách, mang bao nhiêu doanh thu, và nên gọi
nhóm đó là gì để cả đội hiểu?"* Không ai có nhãn sẵn: đây là bài toán **unsupervised**.

Dữ liệu: bộ `online-retail-ii`, sheet `Year 2010-2011` (01/12/2010–09/12/2011), dòng hợp lệ theo định nghĩa dùng chung
(bỏ hoá đơn `C...`, bỏ dòng `Quantity <= 0` hoặc `Price <= 0`, doanh thu gộp), chỉ khách có `Customer ID`. Ngày chốt
(snapshot) là **2011-12-10**, ngày sau giao dịch cuối.

Bạn dựng bảng RFM, đổi thang, chạy k-means, lập hồ sơ cụm và viết một quy tắc **đặt tên** cụm có thể giải thích.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` đọc dữ liệu thật, chạy, vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/14-ml-bigdata-for-analysts/exercises/intermediate/02-rfm-kmeans-segments -q`. Xong thì làm phần **trắc nghiệm** trong app.

In [ ]:
# explore: đọc sheet 2010-2011; lần đầu mất 20-90 giây, sau đó đọc bản parquet trong thư mục tạm
import os
import sys
import tempfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd


def read_retail_1011() -> pd.DataFrame:
    xlsx = ensure("online-retail-ii") / "online_retail_II.xlsx"
    stat = xlsx.stat()
    cache = Path(tempfile.gettempdir()) / "da-cache" / f"retail-1011-{stat.st_size}-{int(stat.st_mtime)}.parquet"
    if cache.exists():
        return pd.read_parquet(cache)
    df = pd.read_excel(xlsx, sheet_name="Year 2010-2011")
    df["Invoice"] = df["Invoice"].astype(str)
    df["StockCode"] = df["StockCode"].astype(str)
    df["Description"] = df["Description"].astype("string")
    cache.parent.mkdir(exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{os.getpid()}.tmp")
    df.to_parquet(tmp)
    tmp.replace(cache)
    return df


tx = read_retail_1011()
print(tx.shape, tx.InvoiceDate.min(), "→", tx.InvoiceDate.max())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

## 1. Bảng RFM

`rfm_table(tx, snapshot) -> DataFrame`: chỉ dòng hợp lệ có `Customer ID` (ép `Invoice` sang `str` trước khi kiểm `C`).
Index `Customer ID`, cột đúng thứ tự:

- `recency`: số ngày trọn từ lần mua cuối tới `snapshot`: `(snapshot - InvoiceDate lớn nhất).days` (mua 12:50 ngày
  2011-12-09 với snapshot 2011-12-10 cho `0`).
- `frequency`: số hoá đơn khác nhau.
- `monetary`: tổng doanh thu gộp.

In [ ]:
def rfm_table(tx: pd.DataFrame, snapshot: pd.Timestamp) -> pd.DataFrame:
    """recency (ngày), frequency (số hoá đơn), monetary (doanh thu) theo Customer ID."""
    invoice = tx["Invoice"].astype(str)
    ok = ~invoice.str.startswith("C") & (tx["Quantity"] > 0) & (tx["Price"] > 0) & tx["Customer ID"].notna()
    lines = tx.loc[ok].assign(Invoice=invoice[ok], revenue=tx.loc[ok, "Quantity"] * tx.loc[ok, "Price"])
    g = lines.groupby("Customer ID")
    out = pd.DataFrame({
        "recency": (pd.Timestamp(snapshot) - g["InvoiceDate"].max()).dt.days,
        "frequency": g["Invoice"].nunique(),
        "monetary": g["revenue"].sum(),
    })
    return out

In [ ]:
# explore: RFM của 4 338 khách, lệch phải rất mạnh
SNAPSHOT = pd.Timestamp("2011-12-10")
rfm = rfm_table(tx, SNAPSHOT)
print(rfm.describe().round(1))

## 2. Đổi thang và k-means

K-means đo khoảng cách, nên cột có số lớn (doanh thu) lấn át cột khác, và vài khách cực lớn kéo tâm cụm.
`prepare_features(rfm) -> DataFrame`: `log1p` cả ba cột, rồi chuẩn hoá mỗi cột về mean 0, độ lệch chuẩn 1 (`ddof=0`).
Giữ index và tên cột.

`fit_kmeans(Z, k, seed=0) -> np.ndarray`: nhãn cụm từ `KMeans(n_clusters=k, n_init=10, random_state=seed)`.

In [ ]:
def prepare_features(rfm: pd.DataFrame) -> pd.DataFrame:
    """log1p rồi chuẩn hoá (ddof=0) từng cột."""
    z = np.log1p(rfm)
    return (z - z.mean()) / z.std(ddof=0)


def fit_kmeans(Z: pd.DataFrame, k: int, seed: int = 0) -> np.ndarray:
    """Nhãn cụm của KMeans(n_clusters=k, n_init=10, random_state=seed)."""
    return KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(Z)

In [ ]:
# explore: chọn k bằng inertia, silhouette, và câu hỏi "đội CRM làm được mấy chương trình?"
from sklearn.metrics import silhouette_score

Z = prepare_features(rfm)
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    print(f"k = {k}: inertia {km.inertia_:,.0f}, silhouette {silhouette_score(Z, km.labels_):.3f}")

# Không đổi thang thì sao? Xem cỡ các cụm
raw_labels = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(rfm)
print("k-means trên RFM thô, cỡ cụm:", np.bincount(raw_labels))

## 3. Hồ sơ cụm

`segment_profile(rfm, labels) -> DataFrame`: index tên `segment` (nhãn cụm), cột đúng thứ tự `n`, `share` (tỉ trọng khách),
`recency_median`, `frequency_median`, `monetary_median`, `revenue_share` (tỉ trọng tổng `monetary`); sắp theo
`monetary_median` giảm dần.

In [ ]:
def segment_profile(rfm: pd.DataFrame, labels) -> pd.DataFrame:
    """n, share, recency_median, frequency_median, monetary_median, revenue_share; sắp monetary_median giảm dần."""
    g = rfm.groupby(pd.Series(np.asarray(labels), index=rfm.index, name="segment"))
    out = pd.DataFrame({
        "n": g.size(),
        "recency_median": g["recency"].median(),
        "frequency_median": g["frequency"].median(),
        "monetary_median": g["monetary"].median(),
        "revenue": g["monetary"].sum(),
    })
    out.insert(1, "share", out["n"] / out["n"].sum())
    out["revenue_share"] = out.pop("revenue") / rfm["monetary"].sum()
    return out.sort_values("monetary_median", ascending=False)

## 4. Đặt tên cụm bằng một quy tắc giải thích được

Thuật toán chỉ trả số. Tên cụm phải nói được **hành vi** so với toàn bộ khách, để ai đọc cũng hiểu vì sao cụm có tên đó.
`name_segments(profile, rfm) -> Series` (index như `profile`, tên `name`), so median của cụm với median của **toàn bộ**
`rfm`:

- *gần đây* khi `recency_median <= ` median recency chung;
- *thường xuyên* khi `frequency_median > ` median frequency chung.

| Gần đây | Thường xuyên | Tên |
| --- | --- | --- |
| có | có | `Champions` |
| không | có | `At risk` |
| có | không | `New / promising` |
| không | không | `Hibernating` |

In [ ]:
SEGMENT_NAMES = {(True, True): "Champions", (False, True): "At risk",
                 (True, False): "New / promising", (False, False): "Hibernating"}


def name_segments(profile: pd.DataFrame, rfm: pd.DataFrame) -> pd.Series:
    """Tên cụm theo bảng SEGMENT_NAMES, so median cụm với median toàn bộ khách."""
    recent = profile["recency_median"] <= rfm["recency"].median()
    frequent = profile["frequency_median"] > rfm["frequency"].median()
    names = [SEGMENT_NAMES[(bool(r), bool(f))] for r, f in zip(recent, frequent)]
    return pd.Series(names, index=profile.index, name="name")

In [ ]:
# explore: hồ sơ k = 4 kèm tên
pd.set_option("display.width", 160)
labels = fit_kmeans(Z, 4)
profile = segment_profile(rfm, labels)
profile["name"] = name_segments(profile, rfm)
print(f"median chung: recency {rfm.recency.median():.0f} ngày, frequency {rfm.frequency.median():.0f}, "
      f"monetary {rfm.monetary.median():,.0f} GBP")
profile.round(3)

In [ ]:
# explore: cụm có ổn định không? Chạy lại với seed khác và so tên của từng khách
named = profile["name"].to_dict()
names_seed0 = pd.Series(labels, index=rfm.index).map(named)
labels_seed7 = fit_kmeans(Z, 4, seed=7)
profile7 = segment_profile(rfm, labels_seed7)
names_seed7 = pd.Series(labels_seed7, index=rfm.index).map(name_segments(profile7, rfm).to_dict())
print(f"khách giữ cùng tên khi đổi seed: {(names_seed0 == names_seed7).mean():.1%}")
print(pd.crosstab(names_seed0, names_seed7))

In [ ]:
# explore: nhìn cụm trên hai trục dễ hiểu (log), mỗi chấm một khách
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
for seg, row in profile.iterrows():
    m = labels == seg
    ax.scatter(rfm.recency[m], rfm.monetary[m], s=6, alpha=0.4, label=f"{row['name']} ({row['n']:,})")
ax.set_yscale("log")
ax.set_xlabel("recency (ngày từ lần mua cuối)")
ax.set_ylabel("doanh thu năm (GBP, log)")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, markerscale=3, fontsize=8)
ax.set_title("4 nhóm khách theo k-means trên RFM", loc="left")
fig.tight_layout()
plt.show()

Viết ra (cho chính bạn, trước khi làm trắc nghiệm):

1. Headline một câu cho Head of CRM: nhóm nào mang phần lớn doanh thu, nhóm nào cần chương trình kéo lại.
2. Vì sao không chọn k = 2 dù silhouette cao nhất?
3. Một việc phải làm trước khi đưa nhãn cụm vào hệ thống CRM (gợi ý: độ ổn định, cách gán khách mới).

Xong thì mở app, làm phần **trắc nghiệm** của bài.